# Tarea 2 -- Preprocesamiento para el modelo de propension

easyMoney - TFM Nuclio Digital School

Este notebook prepara los datos que usaran los notebooks de modelado (uno por producto). Partimos de los 5 CSV en bruto y NO de `df_powerbi.csv` (el CSV de la Tarea 1), por un motivo concreto:

`df_powerbi.csv` tiene una fila por VENTA (resultado de mergear `df_sal` con las demas tablas por `month_sale` + `cid`). Si un cliente no compro nada en un mes, ese mes no aparece.

Para detectar 'este cliente no tenia el producto el mes pasado y ahora si' necesitamos el historico mensual COMPLETO de cada cliente, incluidos los meses sin compra. Eso solo esta en las tablas originales, en concreto en `customer_products.csv`, que es un panel mensual: una fila por cliente y por mes.

## 1. Carga de los CSV en bruto

Solo necesitamos las 3 tablas que tienen estructura de panel mensual (`pk_cid` + `pk_partition`): `customer_products.csv`, `customer_sociodemographics.csv` y `customer_commercial_activity.csv`. `sales.csv` y `product_description.csv` fueron utiles para explorar la estructura de los datos, pero no participan en el panel de modelado.

In [1]:
import pandas as pd
import numpy as np

df_cp  = pd.read_csv('/Users/sandramartinez/Desktop/Proyectos_Data/TFM-EasyMoney/grupo-1-dscesp-0226/data/customer_products.csv', index_col=0)
df_cs  = pd.read_csv('/Users/sandramartinez/Desktop/Proyectos_Data/TFM-EasyMoney/grupo-1-dscesp-0226/data/customer_sociodemographics.csv', index_col=0)
df_cca = pd.read_csv('/Users/sandramartinez/Desktop/Proyectos_Data/TFM-EasyMoney/grupo-1-dscesp-0226/data/customer_commercial_activity.csv', index_col=0)

print('df_cp: ', df_cp.shape)
print('df_cs: ', df_cs.shape)
print('df_cca:', df_cca.shape)

df_cp:  (5962924, 17)
df_cs:  (5962924, 8)
df_cca: (5962924, 6)


## 2. Limpieza de las 3 tablas de panel

Mismos criterios que en la Tarea 1 donde tiene sentido (moda para variables categoricas, mediana para salary), mas dos decisiones nuevas: quitamos `country_id` (si es practicamente siempre 'ES', no aporta informacion) y eliminamos los meses en los que el cliente aparece como fallecido (`deceased`), porque un cliente fallecido no puede ser objetivo de ninguna campana de cross-selling.

In [2]:
# --- Limpieza de df_cp (customer_products) ---
product_cols = ['short_term_deposit','loans','mortgage','funds','securities',
                 'long_term_deposit','em_account_pp','credit_card','payroll',
                 'pension_plan','payroll_account','emc_account','debit_card',
                 'em_account_p','em_acount']

print(df_cp[product_cols].isnull().sum())

# Si el flag de producto es nulo, es que no lo tiene -> 0
df_cp[product_cols] = df_cp[product_cols].fillna(0).astype(int)

short_term_deposit     0
loans                  0
mortgage               0
funds                  0
securities             0
long_term_deposit      0
em_account_pp          0
credit_card            0
payroll               61
pension_plan          61
payroll_account        0
emc_account            0
debit_card             0
em_account_p           0
em_acount              0
dtype: int64


In [3]:
# --- Limpieza de df_cs (customer_sociodemographics) ---
print(df_cs.isnull().sum())

print(df_cs['country_id'].value_counts())
df_cs = df_cs.drop(columns=['country_id'])

print(df_cs['deceased'].value_counts())
df_cs = df_cs[df_cs['deceased'] == 'N'].drop(columns=['deceased'])

# Moda para gender y region_code, mediana para salary (igual que en la Tarea 1)
df_cs['gender'] = df_cs['gender'].fillna(df_cs['gender'].mode()[0])
df_cs['region_code'] = df_cs['region_code'].fillna(df_cs['region_code'].mode()[0])
df_cs['salary'] = df_cs['salary'].fillna(df_cs['salary'].median())

pk_cid                0
pk_partition          0
country_id            0
region_code        2264
gender               25
age                   0
deceased              0
salary          1541104
dtype: int64
country_id
ES    5960672
GB        441
FR        225
DE        199
US        195
CH        194
BR         87
BE         81
VE         79
IE         68
MX         58
AT         51
AR         51
PL         49
IT         45
MA         34
CL         30
CN         28
CA         22
LU         17
ET         17
QA         17
CI         17
SA         17
CM         17
SN         17
MR         17
NO         17
RU         17
CO         17
GA         17
GT         17
DO         17
SE         16
DJ         11
PT         11
JM         11
RO          9
HU          8
DZ          7
PE          4
Name: count, dtype: int64
deceased
N    5961849
S       1075
Name: count, dtype: int64


In [4]:
# --- Limpieza de df_cca (customer_commercial_activity) ---
print(df_cca.isnull().sum())

def moda_segura(x):
    # x.mode() devuelve una lista vacia si TODO el grupo es nulo; en ese caso
    # devolvemos NaN en vez de reventar al pedir el primer elemento
    m = x.mode()
    return m.iloc[0] if not m.empty else np.nan

# Caso 1: falta entry_channel pero hay segment -> moda por segment
mask_solo_canal = df_cca['entry_channel'].isnull() & df_cca['segment'].notnull()
moda_por_segmento = df_cca.groupby('segment')['entry_channel'].agg(moda_segura)
df_cca.loc[mask_solo_canal, 'entry_channel'] = df_cca.loc[mask_solo_canal, 'segment'].map(moda_por_segmento)

# Caso 2: falta segment pero hay entry_channel -> moda por entry_channel
mask_solo_segmento = df_cca['segment'].isnull() & df_cca['entry_channel'].notnull()
moda_por_canal = df_cca.groupby('entry_channel')['segment'].agg(moda_segura)
df_cca.loc[mask_solo_segmento, 'segment'] = df_cca.loc[mask_solo_segmento, 'entry_channel'].map(moda_por_canal)

# Caso 3 + cualquier resto que siga nulo -> Desconocido
df_cca['entry_channel'] = df_cca['entry_channel'].fillna('Desconocido')
df_cca['segment'] = df_cca['segment'].fillna('Desconocido')

df_cca['active_customer'] = df_cca['active_customer'].fillna(0).astype(int)

pk_cid                  0
pk_partition            0
entry_date              0
entry_channel      133033
active_customer         0
segment            133944
dtype: int64


In [5]:
# --- Fechas y orden (en las 3 tablas) ---
# Importante: diff() y shift() dependen del ORDEN de las filas dentro de cada
# cliente. Si no ordenamos por cliente y fecha, el calculo de 'mes anterior'
# saldria mal sin dar ningun error.
for df in [df_cp, df_cs, df_cca]:
    df['pk_partition'] = pd.to_datetime(df['pk_partition'], format='%Y-%m')
    df.sort_values(['pk_cid', 'pk_partition'], inplace=True)

print('df_cp nulos:', df_cp.isnull().sum().sum())
print('df_cs nulos:', df_cs.isnull().sum().sum())
print('df_cca nulos:', df_cca.isnull().sum().sum())

df_cp nulos: 0
df_cs nulos: 0
df_cca nulos: 0


## 3. Construccion del panel cliente-mes

Unimos las tres tablas por `pk_cid` + `pk_partition`. Usamos `inner` con `df_cs` a proposito: como ahi ya quitamos los meses de clientes fallecidos, el inner join hace que esos meses desaparezcan tambien del panel final. Usamos `left` con `df_cca` porque no hemos filtrado nada en esa tabla.

In [6]:
df_panel = df_cp.merge(df_cs, on=['pk_cid', 'pk_partition'], how='inner')
df_panel = df_panel.merge(df_cca, on=['pk_cid', 'pk_partition'], how='left')

print('Forma del panel final:', df_panel.shape)
print('Nulos totales tras el merge:', df_panel.isnull().sum().sum())

df_panel = df_panel.sort_values(['pk_cid', 'pk_partition']).reset_index(drop=True)

Forma del panel final: (5961849, 25)
Nulos totales tras el merge: 0


## 4. Features compartidas: num_products y lag-1

Creamos una feature derivada (`num_products`, cuantos productos tiene el cliente ese mes) y aplicamos `shift(1)` agrupado por cliente a las variables que usaran los dos notebooks de modelado. Esto es la garantia anti-leakage: cada columna `_lag1` representa el valor de esa variable en el MES ANTERIOR para ese mismo cliente, nunca el valor del mes que se quiere predecir.

Incluimos el lag de `pension_plan` y de `em_acount` porque cada notebook de modelado necesita el lag de SU propio producto para filtrar el universo elegible (clientes que el mes anterior no tenian el producto), y el lag del OTRO producto puede usarse como feature (ya que no es constante para ese modelo).

In [7]:
df_panel['num_products'] = df_panel[product_cols].sum(axis=1)

features_compartidas = ['age', 'salary', 'gender', 'segment', 'entry_channel',
                        'active_customer', 'num_products',
                        'pension_plan', 'em_acount']

for col in features_compartidas:
    df_panel[f'{col}_lag1'] = df_panel.groupby('pk_cid')[col].shift(1)

print(df_panel[[f'{c}_lag1' for c in features_compartidas]].isnull().sum())
print('Clientes unicos en el panel:', df_panel['pk_cid'].nunique())

age_lag1                456318
salary_lag1             456318
gender_lag1             456318
segment_lag1            456318
entry_channel_lag1      456318
active_customer_lag1    456318
num_products_lag1       456318
pension_plan_lag1       456318
em_acount_lag1          456318
dtype: int64
Clientes unicos en el panel: 456318


## 5. Guardar el panel para los notebooks de modelado

Guardamos en formato parquet (mas rapido y mucho mas ligero que CSV para casi 6 millones de filas). Si no tienes `pyarrow` instalado: `pip install pyarrow`.

In [9]:
import os
os.makedirs('data', exist_ok=True)

df_panel.to_parquet('data/df_panel_propension.parquet', engine='pyarrow', index=False)
print('Panel guardado en data/df_panel_propension.parquet')
print(df_panel.shape)

Panel guardado en data/df_panel_propension.parquet
(5961849, 35)


In [10]:
df_panel.to_parquet('data/df_panel_propension.parquet', engine='pyarrow', index=False)
print('Panel guardado en data/df_panel_propension.parquet')
print(df_panel.shape)

Panel guardado en data/df_panel_propension.parquet
(5961849, 35)


## 6. Por qué el alcance se queda en pension_plan y em_acount

Antes de dar por cerrado el alcance de la Tarea 2, comprobamos si los otros dos
productos de alto margen identificados en la Tarea 1 (`short_term_deposit` y
`long_term_deposit`) tenían suficiente señal para justificar un modelo propio.

In [11]:
for prod in ['short_term_deposit', 'long_term_deposit']:
    target_tmp = df_panel.groupby('pk_cid')[prod].diff()
    target_tmp = target_tmp.apply(lambda x: 1 if x == 1 else 0)
    lag_tmp = df_panel.groupby('pk_cid')[prod].shift(1)

    elegibles = (lag_tmp == 0)
    total_elegibles = elegibles.sum()
    total_positivos = target_tmp[elegibles].sum()

    print(f'--- {prod} ---')
    print(f'Universo elegible: {total_elegibles}')
    print(f'Adquisiciones nuevas (target=1): {total_positivos}')
    print(f'Tasa de positivos: {total_positivos / total_elegibles * 100:.4f}%')
    print()

# Resultado obtenido:
# short_term_deposit: 2.565 adquisiciones nuevas sobre 5.490.143 elegibles (0.0467%)
# long_term_deposit:  3.997 adquisiciones nuevas sobre 5.411.175 elegibles (0.0739%)
#
# Ambos productos tienen una tasa de positivos 10-15 veces menor que pension_plan
# (0.7%) y em_acount (2.9%). Repartido entre los ~17 meses disponibles, esto deja
# solo ~150-250 adquisiciones reales por mes -- muy pocas para evaluar un modelo
# con fiabilidad (test/val quedarian con una muestra positiva demasiado pequeña).
# Por eso el alcance de la Tarea 2 se queda en pension_plan y em_acount.

--- short_term_deposit ---
Universo elegible: 5490143
Adquisiciones nuevas (target=1): 2565
Tasa de positivos: 0.0467%

--- long_term_deposit ---
Universo elegible: 5411175
Adquisiciones nuevas (target=1): 3997
Tasa de positivos: 0.0739%

